moving avg


In [1]:
"""
Stock Trend Analysis Script (SMA + EMA + Signals)
"""

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import os
from pathlib import Path
import itertools

# ─────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────
CSV_FILES = {
    "HLL":      r"E:\KEC TASK\Download data\HLL Historical Data.csv",
    "TCS":      r"E:\KEC TASK\Download data\TCS Historical Data.csv",
    "RELIANCE": r"E:\KEC TASK\Download data\RELI Historical Data.csv",
    "MSFT":     r"E:\KEC TASK\Download data\MSFT Historical Data.csv",
    "HDBK":     r"E:\KEC TASK\Download data\HDBK Historical Data (1).csv",
}

OUTPUT_DIR = "output_charts_moving_avg"
os.makedirs(OUTPUT_DIR, exist_ok=True)

colors = itertools.cycle(["#2196F3", "#FF5722", "#4CAF50", "#9C27B0", "#FF9800"])

# ─────────────────────────────────────────────
# 2. HELPERS
# ─────────────────────────────────────────────
def parse_volume(vol_str):
    vol_str = str(vol_str).strip().upper().replace(",", "")
    if vol_str in ("", "N/A", "NAN", "-"):
        return np.nan
    multipliers = {"K": 1e3, "M": 1e6, "B": 1e9}
    for suffix, mult in multipliers.items():
        if vol_str.endswith(suffix):
            return float(vol_str[:-1]) * mult
    return float(vol_str)


def load_stock(filepath, ticker):
    if not Path(filepath).exists():
        print(f"[SKIP] {ticker} file not found")
        return None

    try:
        df = pd.read_csv(filepath)
        df.columns = [c.strip().strip('"') for c in df.columns]

        df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
        df = df.dropna(subset=["Date"]).sort_values("Date")

        for col in ["Price", "Open", "High", "Low"]:
            if col in df.columns:
                df[col] = df[col].astype(str).str.replace(",", "").astype(float)

        if "Vol." in df.columns:
            df["Volume"] = df["Vol."].apply(parse_volume)

        if "Change %" in df.columns:
            df["Change_pct"] = df["Change %"].astype(str).str.replace("%", "").astype(float)

        df["Ticker"] = ticker
        print(f"[OK] {ticker}: {len(df)} rows")
        return df

    except Exception as e:
        print(f"[ERROR] {ticker}: {e}")
        return None


# ─────────────────────────────────────────────
# 3. LOAD DATA
# ─────────────────────────────────────────────
stocks = {}
for ticker, path in CSV_FILES.items():
    df = load_stock(path, ticker)
    if df is not None:
        stocks[ticker] = df

if not stocks:
    raise SystemExit("No data loaded")

# ─────────────────────────────────────────────
# 4. INDICATORS (SMA + EMA + SIGNALS)
# ─────────────────────────────────────────────
for ticker, df in stocks.items():
    # SMA
    df["SMA5"] = df["Price"].rolling(5).mean()
    df["SMA10"] = df["Price"].rolling(10).mean()
    df["SMA20"] = df["Price"].rolling(20).mean()

    # EMA
    df["EMA5"] = df["Price"].ewm(span=5, adjust=False).mean()
    df["EMA10"] = df["Price"].ewm(span=10, adjust=False).mean()
    df["EMA20"] = df["Price"].ewm(span=20, adjust=False).mean()

    # Signals
    df["Signal"] = np.where(df["EMA5"] > df["EMA10"], 1, 0)
    df["Crossover"] = df["Signal"].diff()

# ─────────────────────────────────────────────
# 5. CHART: PRICE + SMA + EMA
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

for (ticker, df), color in zip(stocks.items(), colors):
    ax.plot(df["Date"], df["Price"], label=f"{ticker} Price", linewidth=2, color=color)
    ax.plot(df["Date"], df["SMA10"], linestyle="--", alpha=0.6, color=color)
    ax.plot(df["Date"], df["EMA10"], linestyle="-.", linewidth=1.5, color=color)

ax.set_title("Price with SMA & EMA")
ax.legend()
ax.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "price_sma_ema.png"))
plt.close()

# ─────────────────────────────────────────────
# 6. INDIVIDUAL STOCK WITH SIGNALS
# ─────────────────────────────────────────────
for (ticker, df), color in zip(stocks.items(), colors):

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(df["Date"], df["Price"], label="Price", color=color)
    ax.plot(df["Date"], df["EMA5"], label="EMA5", linestyle="-")
    ax.plot(df["Date"], df["EMA10"], label="EMA10", linestyle="--")

    # Buy signals
    buy = df[df["Crossover"] == 1]
    sell = df[df["Crossover"] == -1]

    ax.scatter(buy["Date"], buy["Price"], marker="^", color="green", s=80, label="BUY")
    ax.scatter(sell["Date"], sell["Price"], marker="v", color="red", s=80, label="SELL")

    ax.set_title(f"{ticker} Trading Signals")
    ax.legend()
    ax.grid(True)

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{ticker}_signals.png"))
    plt.close()

# ─────────────────────────────────────────────
# 7. SAVE COMBINED DATA
# ─────────────────────────────────────────────
all_data = pd.concat(stocks.values())
all_data.to_csv(os.path.join(OUTPUT_DIR, "final_output.csv"), index=False)

print("\n✅ DONE — SMA, EMA, and signals added successfully!")

[OK] HLL: 20 rows
[OK] TCS: 20 rows
[OK] RELIANCE: 20 rows
[OK] MSFT: 21 rows
[OK] HDBK: 834 rows

✅ DONE — SMA, EMA, and signals added successfully!


Moving Averages (Trend Detection)
 SMA5, SMA10, SMA20
 Smooths price fluctuations
 Shows trend direction

CORE LOGIC — TRADING SIGNALS
Signal = EMA5 > EMA10
Crossover = Signal.diff()

| Condition    | Signal    |
| ------------ | --------- |
| EMA5 > EMA10 | Uptrend   |
| EMA5 < EMA10 | Downtrend |



Buy Signal
Crossover == 1
EMA5 crosses above EMA10

Sell Signal
Crossover == -1
EMA5 crosses below EMA10



VISUALIZATION
Shows:

Price trend
SMA (smooth trend)
EMA (fast trend)



Individual Stock Signals
Price
EMA5 & EMA10
▲ Buy markers
▼ Sell markers



OUTPUT
price_sma_ema.png
<ticker>_signals.png


final_output.csv

output will generate in output_charts_moving_avg this  folder


In this module, I implemented technical indicators like SMA and EMA to analyze stock trends. I used EMA crossover logic to generate buy and sell signals, where a short-term EMA crossing above a long-term EMA indicates a potential buying opportunity. I then visualized these signals on price charts to make the strategy interpretable.